In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

import kovariance

In [ ]:

data_file = r"C:\DATA\StonyBrookCollab\2025_10_01_tpx\C10F18\pymepix_C10F18_test2_2025-10-01_13-03_CENT.csv"

df_orig = pd.read_csv(data_file)


In [ ]:
kovariance.timewalk_summary(tof=df_orig["#tof"].to_numpy() * 1e6, tot=df_orig["#atot"].to_numpy(),
                            tofbins=np.linspace(4, 4.5, 100), totbins=np.linspace(0, 800, 100))
timewalk_fit = kovariance.get_timewalk_fit(
        tof=df_orig["#tof"].to_numpy() * 1e6,
        tot=df_orig["#atot"].to_numpy(),
        tofrange=(4.25, 4.4),
        totrange=(50, 600),
)

tof_corr = timewalk_fit(df_orig["#tof"].to_numpy() * 1e6)

kovariance.timewalk_summary(
        tof=tof_corr,
        tot=df_orig["#atot"].to_numpy(),
        tofbins=np.linspace(4.2, 4.5, 100),
        totbins=np.linspace(0, 800, 100),
        title="After Timewalk Correction"
)
df_tw = df_orig.copy()
df_tw["#tof"] = tof_corr / 1e6

In [ ]:
df = df_orig.copy()
df.rename(columns={"#tof": "tof", "#x": "x", "#y": "y", "#trig": "trig"}, inplace=True)
df["tof"] /= 1e-6  # convert to us
df["r"] = np.sqrt((df['x'] - 138) ** 2 + (df['y'] - 133) ** 2)
idx = (df['tof'] > 4.5) & (df['tof'] < 13) & (df['x'] > 0) & (df['x'] < 256) & (df['y'] > 0) & (
            df['y'] < 256)  #& (df['r']<25)

df_u = df.copy()
df = df[idx]
px.density_heatmap(
        df,
        x="x", y="y",
        nbinsx=256, nbinsy=256,
        title="Propylene Oxide",
        width=600, height=600,
        color_continuous_scale=px.colors.sequential.Inferno,
        range_color=[0, 500]
).show()

px.histogram(df, x="tof", title="PFD ToF", width=600, height=400, log_y=True, nbins=10000).show()

hist, tof_edges, r_edges = np.histogram2d(df['tof'], df['r'], bins=[1000, 100], range=[[4.5, 13], [0, 120]])

hist *= r_edges[1:]  # scale by r^2 to account for area increase
hist = hist + 1  # to avoid log(0)
# hist = np.clip(hist, 1, np.percentile(hist, 99))  # clip extreme values for better color scaling
px.imshow(
        np.log(hist.T),
        x=tof_edges[:-1],
        y=r_edges[:-1],
        aspect="auto",
        origin="lower",
        title="ToF vs Radius",
        width=600,
        height=400,
        labels={"x": "ToF (us)", "y": "Radius (pixels)", "color": "Counts"},
        color_continuous_scale=px.colors.sequential.Inferno,

).show()

In [ ]:
# calibration_points = [(4.24, 0), (4.67,1), (4.82,2), (5.76, 18), (7.2,58), (4.664,1), (4.828,2), (5.77,18), (5.621, 16), (7.713, 80)]
# calibration_points = [(4.24, 0), (4.67,1), (4.82,2), (5.57, 19), (5.91,31), (7.7,150), (7.2, 112)]
calibration_points = [(7.71, 80), (5.77, 15), (6.68, 39)]
x_cal, y_cal = zip(*calibration_points)

fit = np.polyfit(x_cal, y_cal, 2)
p = np.poly1d(fit)
print(f"Calibration fit: {p}")
x_sample = np.linspace(4, 8, 100)
y_sample = p(x_sample)

px.line(x=x_sample, y=y_sample, title="ToF Calibration Curve", width=600, height=400,
        labels={"x": "ToF (us)", "y": "m/z"}).add_scatter(x=x_cal, y=y_cal, mode="markers",
                                                          name="Calibration Points").show()
print(p(6.31))

df['mz'] = p(df['tof'])

hist, tof_edges, r_edges = np.histogram2d(df['mz'], df['r'], bins=[400, 50], range=[[0, 200], [0, 125]])

# hist*= r_edges[1:]  # scale by r^2 to account for area increase
hist = hist + 1  # to avoid log(0)
# hist = np.clip(hist, 1, np.percentile(hist, 99))  # clip extreme values for better color scaling
px.imshow(
        np.log10(hist.T),
        x=tof_edges[:-1],
        y=r_edges[:-1],
        aspect="auto",
        origin="lower",
        title="ToF vs Radius",
        width=600,
        height=400,
        labels={"x": "m/q", "y": "Radius (pixels)", "color": "log_10 Counts"},
        color_continuous_scale=px.colors.sequential.Inferno,
).show()

px.histogram(
        df, x="mz", title="PFD m/q", width=900, height=400, log_y=True, nbins=10000, range_x=[0, 200],
        range_y=[1, 100000],
).show()


In [ ]:
import skimage

hist, tof_edges, r_edges = np.histogram2d(df['mz'], df['r'], bins=[500, 500], range=[[0, 200], [0, 125]])

px.line(x=tof_edges[1:], y=np.log10(np.sum(hist, axis=1)) * tof_edges[1:] ** 0.4, title="m/q Spectrum", width=600,
        height=400, labels={"x": "m/q", "y": "log_10 Counts"}).show()
dog = skimage.filters.difference_of_gaussians(np.log(hist.T + 1) * tof_edges[1:] ** 0.4, low_sigma=0.5, high_sigma=1000)

fig = px.imshow(
        dog,
        x=tof_edges[:-1],
        y=r_edges[:-1],
        aspect="auto",
        origin="lower",
        title="ToF vs Radius",
        width=1200,
        height=800,
        labels={"x": "m/q", "y": "Radius (pixels)", "color": "log_10 Counts"},
        color_continuous_scale=px.colors.sequential.Inferno,
)
#
for c in range(10):
    for f in range(18):
        mz = 12 * c + 19 * f
        if f > 2 * c + 2 or mz > 200:
            continue
        print(f"C{c}F{f}: {12 * c + 19 * f}")
        fig.add_vline(x=mz, line_dash="dash", line_color="white", annotation_text=f"C{c}F{f}",
                      annotation_position="top left", annotation_font_color="white")
fig.show()


In [ ]:
fragments = (
    (1 / 2, 0, "C(2+)"),
    (1, 0, "C+"),
    (0, 1, "F+"),
    (2, 0, "C2+"),
    (3 / 2, 1 / 2, "C3F(2+)"),
    (1, 1, "CF+"),
    (2, 1, "C2F+"),
    (0, 2, "F2+"),
    (2, 2, "C2F2+"),
    (5 / 2, 3 / 2, "C5F3(2+)"),
    (1, 2, "CF2+"),
    (1, 3, "CF3+"),
    (3, 2, "C3F2+"),
    (2, 3, "C2F3+"),
    (3, 3, "C3F3+"),
    (2, 4, "C2F4+"),
    (3, 4, "C3F4+"),
    (2, 5, "C2F5+"),
    (4, 4, "C4F4+"),
    (3, 5, "C3F5+"),
    (4, 5, "C4F5+"),
    (3, 6, "C3F6+"),
    (4, 7, "C4F7+"),
    (6, 6, "C6F6+"),
)

fig = px.imshow(
        dog,
        x=tof_edges[:-1],
        y=r_edges[:-1],
        aspect="auto",
        origin="lower",
        title="ToF vs Radius",
        width=1600,
        height=800,
        labels={"x": "m/q", "y": "Radius (pixels)", "color": "log_10 Counts"},
        color_continuous_scale=px.colors.sequential.Inferno,
)
for i, (c, f, label) in enumerate(fragments):
    mz = 12 * c + 19 * f
    if mz > 200:
        continue
    print(f"C{c}F{f}: {12 * c + 19 * f}")
    fig.add_vline(x=mz, line_dash="dash", line_color="white", annotation_text=label, annotation_position="top left",
                  annotation_font_color="white", annotation_yshift=-10 * (i % 2))

fig.show()

In [ ]:
cov_bins = np.linspace(0, 200, 201)
cov_matrix = np.zeros((len(cov_bins) - 1, len(cov_bins) - 1))
base_matrix = np.zeros((len(cov_bins) - 1, len(cov_bins) - 1))

df_filt = df[df['mz'] > 4]
groups = df_filt.groupby("trig").filter(lambda x: len(x) > 1).groupby("trig")
import itertools

for (_, group), (_, next_group) in itertools.pairwise(groups):
    mz_values = group['mz'].values
    for mz1, mz2 in itertools.combinations(mz_values, 2):
        bin1 = np.digitize(mz1, cov_bins) - 1
        bin2 = np.digitize(mz2, cov_bins) - 1
        if 0 <= bin1 < len(cov_bins) - 1 and 0 <= bin2 < len(cov_bins) - 1:
            cov_matrix[bin1, bin2] += 1
            cov_matrix[bin2, bin1] += 1  # symmetric

    mz_next = next_group['mz'].values
    for mz1, mz2 in itertools.product(mz_values, mz_next):
        bin1 = np.digitize(mz1, cov_bins) - 1
        bin2 = np.digitize(mz2, cov_bins) - 1
        if 0 <= bin1 < len(cov_bins) - 1 and 0 <= bin2 < len(cov_bins) - 1:
            base_matrix[bin1, bin2] += 1
            base_matrix[bin2, bin1] += 1  # symmetric

# for i in range(len(cov_bins)-1):
#     cov_matrix[i,i]=0  # zero out diagonal

In [ ]:
cov_matrix_norm = cov_matrix  #/np.sum(cov_matrix)*1000000
base_matrix_norm = base_matrix  #/np.sum(base_matrix)*1000000
data_log = np.where(cov_matrix_norm - base_matrix_norm >= 0, np.log10(cov_matrix_norm - base_matrix_norm + 1),
                    -np.log10(base_matrix_norm - cov_matrix_norm + 1))
px.imshow(
        np.log(base_matrix_norm + 1),
        x=cov_bins[:-1],
        y=cov_bins[:-1],
        aspect="auto",
        origin="lower",
        title="i, i+1 Baseline Matrix of m/q",
        width=800,
        height=800,
        labels={"x": "m/q", "y": "m/q", "color": "log_10 Counts"},
        color_continuous_scale=px.colors.sequential.Inferno,
).show()
px.imshow(
        np.log(cov_matrix_norm + 1),
        x=cov_bins[:-1],
        y=cov_bins[:-1],
        aspect="auto",
        origin="lower",
        title="i, i Matrix of m/q",
        width=800,
        height=800,
        labels={"x": "m/q", "y": "m/q", "color": "log_10 Counts"},
        color_continuous_scale=px.colors.sequential.Inferno,
).show()

fig = px.imshow(
        data_log,
        x=cov_bins[:-1],
        y=cov_bins[:-1],
        aspect="auto",
        origin="lower",
        title="Covariance Matrix of m/q",
        width=1600,
        height=1600,
        labels={"x": "m/q", "y": "m/q", "color": "log_10 Counts"},
        color_continuous_scale=px.colors.sequential.Bluered,
        color_continuous_midpoint=0
)

for i, (c, f, label) in enumerate(fragments):
    mz = 12 * c + 19 * f
    if mz > 200:
        continue
    print(f"C{c}F{f}: {12 * c + 19 * f}")
    fig.add_vline(x=mz, line_dash="dash", line_color="white", annotation_text=label, annotation_position="top left",
                  annotation_font_color="white", annotation_yshift=-10 * (i % 2))
    fig.add_hline(y=mz, line_dash="dash", line_color="white", annotation_text=label, annotation_position="top right",
                  annotation_font_color="white")
fig.show()
fig.write_image("covariance_matrix_mq.png", scale=2)

In [ ]:
print(df.groupby("trig").size().describe())
g = df.groupby("trig")
filt = g.size() > 1
df_filt = g.filter(lambda x: len(x) > 1)
print(df_filt.groupby("trig").size().describe())

In [ ]:
import kovariance

cov_result = kovariance.self_covariance(df['trig'].to_numpy(), df['mz'].to_numpy(), np.linspace(0, 200, 200))

In [ ]:
cov_result.PlotSummary(logscale=True)

pos_cov = cov_result.poscov
neg_cov = cov_result.negcov

fig = px.imshow(np.log(pos_cov * 1e6 + 1), origin="lower", width=1600, height=1600,
                labels={"x": "m/q", "y": "m/q", "color": "log Counts"},
                color_continuous_scale=px.colors.sequential.Inferno,
                title="Positive Covariance Matrix of m/q")
fig.update_xaxes(range=[0, 100])
fig.update_yaxes(range=[0, 100], autorange=False)
for i, (c, f, label) in enumerate(fragments):
    mz = 12 * c + 19 * f
    if mz > 200:
        continue
    print(f"C{c}F{f}: {12 * c + 19 * f}")
    fig.add_vline(x=mz, line_dash="dash", line_color="white", annotation_text=label, annotation_position="top left",
                  annotation_font_color="white", annotation_yshift=-10 * (i % 2))
    fig.add_hline(y=mz, line_dash="dash", line_color="white", annotation_text=label, annotation_position="top right",
                  annotation_font_color="white")

fig.show()
fig.write_image("covariance_matrix_mq_pos.png", scale=2)